In [ ]:
from platform import python_version
print(python_version())

### BayesPrism

https://github.com/Danko-Lab/BayesPrism


#### **Bayesian cell Proportion Reconstruction** Inferred using Statistical Marginalization (BayesPrism):

A Fully Bayesian Inference of Tumor Microenvironment composition and gene expression

BayesPrism consists of 
- the deconvolution modules and 
- the embedding learning module. 

The **deconvolution module** models a prior from cell type-specific expression profiles from scRNA-seq to jointly estimate the posterior distribution of cell type composition and cell type-specific gene expression from bulk RNA-seq expression of tumor (or non-tumor) samples. 

The **embedding learning** module uses Expectation-maximization (EM) to approximate the tumor expression using a linear combination of malignant gene programs while conditional on the inferred expression and fraction of non-malignant cells estimated by the deconvolution module.


#### Ref

Cell type and gene expression deconvolution with BayesPrism enables Bayesian integrative analysis across bulk and single-cell RNA sequencing in oncology

Chu, T. et al. & Danko, C.G.

https://www.nature.com/articles/s43018-022-00356-3


#### Concepts (paper)

Two layers of information are critical for understanding tumor composition: (1) the proportion of each cell type and (2) the levels of gene expression in each cell type. The rise of single-cell RNA sequencing (scRNA-seq) technologies has recently enabled direct, genome-wide measurement of the transcriptome in individual
cells within the TME and characterization of their heterogeneity. However, the cost of scRNA-seq and requirements for high-quality tissue limit the number of patient samples that can be assayed9. Moreover, **scRNA-seq is susceptible to technical biases in cell capture** 9 , which confound the recovery of cell type composition.

### Github

https://github.com/Danko-Lab/BayesPrism

- tutorial_deconvolution.html
- tutorial_embedding_learning.html



In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

In [ ]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

### Open primary cites from cbio

In [ ]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

verbose=True

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

In [ ]:
verbose=False
force=False

imax_tumor=200
imax_normal=100

df_tumor, df_normal, df_gtex_ctrl = cbio.calc_file_expression_tumor_normal_gtex(
            imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)

print(df_tumor.shape[1], df_normal.shape[1], df_gtex_ctrl.shape[1])


In [ ]:
df_tumor.head(3)

In [ ]:
df_normal.head(3)

In [ ]:
df_gtex_ctrl.head(3)

### All samples

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

In [ ]:
dfn_tumor.head(3)

In [ ]:
dfn_normal.head(3)

In [ ]:
df_gtex_ctrl.head(3)

In [ ]:
cbio.plot_boxplot_expression(dfn_tumor, do_log10=True, title = "Expression across tumor samples")

In [ ]:
cbio.plot_boxplot_expression(dfn_normal, do_log10=True, title = "Expression across normal samples")

### Prism - development

In [ ]:
import anndata as ad

from libs.prism_lib import PRISM

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

In [ ]:
fname="count-matrix.txt"
prism.load_and_view_matrix_txt(fname=fname, nrows=10)

In [ ]:
adata = prism.load_matrix(fname=fname)
fname_celltype="all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype)

In [ ]:
rep = prism.check_reference(adata=adata)
assert rep["X_looks_like_raw_counts"], rep["problems"]

In [ ]:
ref, s2t = prism.pseudobulk_reference(adata)

In [ ]:
df_bulk, meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata)
gene_subset=prism.select_genes(ref)


In [ ]:
meta_desc = dict(reference="Peng2019_CRA001160",
              cohorts=["TCGA-PAAD", "CPTAC3"],
              strand="unstranded",
              method="InstaPrism")

force=False
verbose=True

res = prism.run_bayesprism(df_bulk=df_bulk, meta_desc=meta_desc, 
                           ref=ref, state_to_type=s2t, 
                           gene_subset=gene_subset,
                           force=force, verbose=verbose)

### Cell state (Tutorial: bulk RNA-seq deconvolution using BayesPrism)

Please make sure that all cell states contain a reasonable number of cells, e.g. >20 or >50, so that their profile can be represented accurately.

What to supply for cell.state.labels and cell.type.labels? The definition of cell type and cell state can be somewhat arbitrary (similar to the issue of assigning cell types for scRNA-seq) and depends on the question of interest. Their definitions depend on the granularity we aim at and the confidence of the cell.type.labels in scRNA-seq data. Usually, a good rule of thumb is as follows. 1) Define cell types as the cluster of cells having a sufficient number of significantly differentially expressed genes than other cell types, e.g., greater than 50 or even 100. For clusters that are too similar in transcription, we recommend treating them as cell states, which will be summed up before the final Gibbs sampling. Therefore, cell states are often suitable for cells that form a continuum on the phenotypic manifold rather than distinct clusters. 2) Define multiple cell states for cell types of significant heterogeneity, such as malignant cells, and of interest to deconvolve their transcription.

In [ ]:
s2t

In [ ]:
dic = res.__dict__
dic.keys()

In [ ]:
res.states

In [ ]:
res.tumor_purity

In [ ]:
res.theta_type

In [ ]:
res.theta_type.loc['T-C3L-02890'].sum()

In [ ]:
res.theta

In [ ]:
res.Z.shape

In [ ]:
len(res.genes)

### Prism

load_cra001160.py  

Convert the Peng 2019 GSA deposit into an AnnData ready for
`paad_deconv.pseudobulk_reference()`.

Input (from ftp://download.big.ac.cn/gsa/CRA001160/):
- count-matrix.txt   2.77 GB dense TSV, genes x cells
- all_celltype.txt   2.1 MB, per-cell annotation

The matrix is dense text: ~20k genes x ~57k cells is ~1.1e9 values, which is 10-13 GB as a dense float array but well under 1 GB as CSR, since scRNA counts are >90% zeros. So it is parsed in row chunks and sparsified incrementally -- never materialised dense.

```Bash
lftp -e "cls -l; quit" ftp://download.big.ac.cn/gsa/CRA001160/

lftp -e "pget -n 8 -c count-matrix.txt; \
         get all_celltype.txt; get md5sum.txt; quit"      ftp://download.big.ac.cn/gsa/CRA001160/
```


In [ ]:
!python -m ipykernel install --user --name renv --display-name "Python (renv)"

### Single-cell quality: MAESTRO

MAESTRO (Model-based AnalysEs of Single-cell Transcriptome and RegulOme) is a Snakemake-based pipeline that processes single-cell RNA-seq and ATAC-seq data from raw FASTQ files through alignment, quality control, cell filtering, clustering, and cell-type annotation.

https://liulab-dfci.github.io/MAESTRO/

### TISCH2

https://tisch.compbio.cn/gallery/?cancer=PAAD&celltype=Acinar&celltype=Ductal&species=Human&treatment=None&primary=Primary


ref: Peng J, Sun BF, Chen CY, Zhou JY, Chen YS, Chen H, Liu L, Huang D, Jiang J, Cui GS, Yang Y, Wang W, Guo D, Dai M, Guo J, Zhang T, Liao Q, Liu Y, Zhao YL, Han DL, Zhao Y, Yang YG, Wu W. Single-cell RNA-seq highlights intra-tumoral heterogeneity and malignant progression in pancreatic ductal adenocarcinoma. Cell Res. 2019 Sep;29(9):725-738. doi: 10.1038/s41422-019-0195-y. Epub 2019 Jul 4. Erratum in: Cell Res. 2019 Sep;29(9):777. doi: 10.1038/s41422-019-0212-1. PMID: 31273297; PMCID: PMC6796938.

In [ ]:
import anndata as ad

# sc_ref = ad.read_h5ad("PAAD_Peng2019_annotated.h5ad")   # raw counts in .X

###  nnls_deconvolve()

It's the baseline cross-check — deliberately not part of the main path. It's there so you can ask "is my reference sane?" without trusting the engine you're validating.

What it computes. For each sample independently, it solves

min_w  ||Φᵀw − b_s||²    subject to  w ≥ 0

where b_s is the sample's CPM vector and Φᵀ is the genes × states signature matrix (each state row CPM-normalized). Then it rescales w to sum to 1. That's the dtangle / CIBERSORT family: linear unmixing under a Gaussian loss.

How it differs from prism_em, which matters more than it looks:

|	      | nnls_deconvolve	 |prism_em |
|---------|------------------|---------|
| loss	  | L2 on CPM	| multinomial on counts |
| gene weighting | high-expression genes dominate | Poisson variance weights each gene naturally |
| simplex	|  imposed post hoc by rescaling | enforced every iteration |
| malignant reference | fixed | sample-specific (stage 2) |

The L2-on-CPM part is the substantive difference. A gene at 5,000 CPM contributes ~10⁶× more residual than one at 5 CPM, so NNLS is effectively fit on a few dozen highly-expressed genes regardless of how informative they are. The multinomial likelihood weights each gene by its own expected count, which is the correct variance model for counts.

Why you'd actually run it. Concordance is a reference-quality diagnostic, and the pattern of disagreement is informative:

Stromal/immune states agree closely (Spearman > 0.8) → reference is fine
Stromal/immune states disagree → your phi is broken, or select_genes picked protocol-driven genes; fix that before interpreting anything
Malignant compartment disagrees while the rest agrees → expected and good. That's stage 2 doing its job. If NNLS and EM agree on purity, update_malignant_reference isn't contributing and PDAC classical/basal heterogeneity is still leaking into the stromal fractions

To wire it in:

In [ ]:
th_nnls = prism.nnls_deconvolve(df_bulk, ref, genes=res.genes)
th_nnls

In [ ]:
th_em   = res.theta
th_em

In [ ]:
conc = pd.DataFrame({
    "spearman": {k: th_em[k].corr(th_nnls[k], method="spearman") for k in res.states},
    "mean_em":   th_em.mean(),
    "mean_nnls": th_nnls.mean(),
})
conc["bias"] = conc.mean_em - conc.mean_nnls
print(conc.sort_values("spearman"))

Pass genes=res.genes explicitly. The default (all shared genes) lets housekeeping genes drive the L2 fit and the comparison stops being meaningful.

One caveat before you trust a low correlation. 

NNLS handles collinear reference profiles badly 
— with Ductal cell type 1 vs type 2, or iCAF vs myCAF, 

it tends to zero one of the pair out arbitrarily per sample, so its per-state estimates are unstable even when the aggregate is right. 

Check the conditioning first:

In [ ]:
g = df_bulk.index.intersection(ref.columns).intersection(pd.Index(res.genes))
len(g), g

In [ ]:
Phi = (ref[g].div(ref[g].sum(axis=1), axis=0)).T.to_numpy()
Phi

In [ ]:
print("cond(Phi):", np.linalg.cond(Phi))
pd.DataFrame(np.corrcoef(Phi.T), index=res.states, columns=res.states).round(2)

Condition number above ~10³ means the states aren't separable from this reference and the NNLS disagreement is telling you about the reference, not the engine. 

Comparing at the coarse type level (res.theta_type) rather than the fine state level sidesteps this and is usually the fairer test.